# Optuna study analysis — TCN hyperparameter search

Loads the stage 2 JournalFile study and visualises:
- Study overview and best parameters
- Optimisation history
- Learning curves (best trial + top-N comparison)
- Hyperparameter distributions vs objective
- Parameter importances
- Parallel coordinates

In [ ]:
STUDY_LOG    = "data/output/gait/stage2/study.log"
STUDY_NAME   = "tcn_joint"
TOP_N        = 5   # top trials to show in learning-curve comparison

LOAO_DIR     = "data/output/gait/stage4/results"  # loao_trial_<id>.json files

In [ ]:
import os

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
storage = JournalStorage(JournalFileBackend(STUDY_LOG))
study   = optuna.load_study(study_name=STUDY_NAME, storage=storage)

all_trials      = study.trials
complete_trials = [t for t in all_trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned_trials   = [t for t in all_trials if t.state == optuna.trial.TrialState.PRUNED]

print(f"Total trials    : {len(all_trials)}")
print(f"Complete        : {len(complete_trials)}")
print(f"Pruned          : {len(pruned_trials)}")
print(f"Best val loss   : {study.best_value:.4f}  (trial #{study.best_trial.number})")
print(f"Best params     : {study.best_params}")

## Optimisation history

In [ ]:
trial_nums  = [t.number for t in complete_trials]
objectives  = [t.value  for t in complete_trials]
best_so_far = pd.Series(objectives).cummin().tolist()

fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(trial_nums, objectives, s=18, alpha=0.6, color="#4e79a7", label="trial val loss")
ax.plot(trial_nums, best_so_far, color="#e15759", linewidth=2, label="best so far")
ax.axhline(study.best_value, color="#e15759", linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("Trial")
ax.set_ylabel("Val loss (NLL)")
ax.set_title("Optimisation history")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Learning curve — best trial

In [ ]:
cutoff_epoch = 5

best = study.best_trial
train_losses = best.user_attrs.get("train_losses", [])
val_losses   = best.user_attrs.get("val_losses",   [])
best_epoch   = best.user_attrs.get("best_epoch",   None)

if not train_losses:
    print("No loss history stored for the best trial.")
else:
    epochs = range(1, len(train_losses) + 1)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(epochs, train_losses, label="train",      color="#4e79a7")
    ax.plot(epochs, val_losses,   label="validation", color="#f28e2b")
    if best_epoch:
        ax.axvline(best_epoch, color="grey", linestyle="--", linewidth=1,
                   label=f"best epoch ({best_epoch})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("NLL loss")
    ax.set_title(f"Best trial #{best.number} — learning curve  "
                 f"(val loss {best.value:.4f})")
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xlim(epochs[cutoff_epoch], epochs[-1])
    ax.set_ylim(top=max(max(train_losses[cutoff_epoch:]), max(val_losses[cutoff_epoch:])) * 1.1)
    plt.tight_layout()
    plt.show()
    print(f"Params: {best.params}")

## Learning curves — top-N trials

In [ ]:
trials_with_history = [
    t for t in complete_trials if "val_losses" in t.user_attrs
]
top_trials = sorted(trials_with_history, key=lambda t: t.value)[:TOP_N]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = cm.tab10(np.linspace(0, 0.9, len(top_trials)))

for ax_train, ax_val, label in zip(
    [axes[0]] * len(top_trials),
    [axes[1]] * len(top_trials),
    ["train", "val"],
):
    break  # just need ax setup

for trial, color in zip(top_trials, colors):
    tl = trial.user_attrs["train_losses"]
    vl = trial.user_attrs["val_losses"]
    ep = range(1, len(tl) + 1)
    lbl = f"#{trial.number} ({trial.value:.4f})"
    axes[0].plot(ep, tl, color=color, linewidth=1.2, label=lbl)
    axes[1].plot(ep, vl, color=color, linewidth=1.2, label=lbl)

for ax, title in zip(axes, ["Train loss", "Val loss"]):
    ax.set_xlabel("Epoch")
    ax.set_ylabel("NLL loss")
    ax.set_title(f"Top {TOP_N} trials — {title}")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Hyperparameter distributions vs objective

In [ ]:
df = pd.DataFrame(
    [{**t.params, "val_loss": t.value} for t in complete_trials]
)

# Use Optuna distributions to decide plot type — more reliable than dtype heuristics.
dists      = complete_trials[0].distributions
param_names = list(study.best_params.keys())
n_params   = len(param_names)
ncols = 3
nrows = (n_params + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.array(axes).flatten()

for ax, param in zip(axes, param_names):
    dist = dists[param]
    vals = df[param]

    if isinstance(dist, optuna.distributions.CategoricalDistribution):
        categories = sorted(vals.unique(), key=str)
        data = [df.loc[df[param] == c, "val_loss"].values for c in categories]
        ax.boxplot(data, labels=[str(c) for c in categories])
        ax.tick_params(axis="x", rotation=15)
    else:
        sc = ax.scatter(vals, df["val_loss"], c=df["val_loss"],
                        cmap="RdYlGn_r", s=20, alpha=0.7)
        plt.colorbar(sc, ax=ax, label="val loss")
        if isinstance(dist, optuna.distributions.FloatDistribution) and dist.log:
            ax.set_xscale("log")

    ax.set_ylabel("val loss")
    ax.set_xlabel(param)
    ax.set_title(param)
    ax.grid(alpha=0.3)

for ax in axes[n_params:]:
    ax.set_visible(False)

plt.suptitle("Hyperparameter distributions vs validation loss", y=1.01)
plt.tight_layout()
plt.show()

## Parameter importances

In [ ]:
importances = optuna.importance.get_param_importances(study)

params_sorted = list(importances.keys())
values_sorted = list(importances.values())

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(params_sorted, values_sorted, color="#4e79a7", alpha=0.85)
ax.bar_label(bars, fmt="{:.3f}", padding=3, fontsize=9)
ax.set_xlabel("Importance (fANOVA)")
ax.set_title("Hyperparameter importances")
ax.set_xlim(0, max(values_sorted) * 1.2)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## Parallel coordinates

In [ ]:
# Normalise each parameter to [0, 1] for parallel-coordinates display.
# Colour encodes val_loss (green = low = good).
# String categoricals are mapped to integers; their label mapping is annotated on the axis.

df_pc = df.copy()

# Build label maps for all categorical params; store for axis annotation.
cat_label_maps: dict[str, list[str]] = {}
for col in param_names:
    dist = dists[col]
    if isinstance(dist, optuna.distributions.CategoricalDistribution):
        cats = sorted(df_pc[col].unique(), key=str)
        cat_label_maps[col] = [str(c) for c in cats]
        df_pc[col] = df_pc[col].map({c: i for i, c in enumerate(cats)})

cols_to_plot = param_names + ["val_loss"]
df_norm = df_pc[cols_to_plot].copy()
for col in cols_to_plot:
    mn, mx = df_norm[col].min(), df_norm[col].max()
    df_norm[col] = (df_norm[col] - mn) / (mx - mn + 1e-12)

cmap   = cm.RdYlGn_r
colors = cmap(df_pc["val_loss"].rank(pct=True).values)

fig, ax = plt.subplots(figsize=(max(13, len(cols_to_plot) * 1.8), 5))
x = np.arange(len(cols_to_plot))

for i, row in df_norm.iterrows():
    ax.plot(x, row[cols_to_plot].values, color=colors[i], alpha=0.4, linewidth=0.8)

best_row = df_norm.iloc[df["val_loss"].idxmin()]
ax.plot(x, best_row[cols_to_plot].values, color="red", linewidth=2.5,
        label=f"best trial #{study.best_trial.number}", zorder=5)

ax.set_xticks(x)
ax.set_xticklabels(cols_to_plot, rotation=20, ha="right")
ax.set_ylabel("Normalised value")
ax.set_title("Parallel coordinates (colour = val loss, red = best trial)")
ax.legend()

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(df["val_loss"].min(),
                                                          df["val_loss"].max()))
plt.colorbar(sm, ax=ax, label="val loss")

# Annotate categorical axes with their label mapping (0 = ..., 1 = ...).
for col, labels in cat_label_maps.items():
    xi = cols_to_plot.index(col)
    mapping_str = "  ".join(f"{i}={lbl}" for i, lbl in enumerate(labels))
    ax.text(xi, -0.18, mapping_str, ha="center", va="top", fontsize=7,
            transform=ax.get_xaxis_transform(), color="#555555")

plt.tight_layout()
plt.show()

## Summary table

In [ ]:
rows = []
for t in sorted(complete_trials, key=lambda t: t.value):
    row = {"trial": t.number, "val_loss": round(t.value, 4)}
    row.update(t.params)
    row["epochs"] = len(t.user_attrs.get("val_losses", []))
    row["best_epoch"] = t.user_attrs.get("best_epoch", None)
    rows.append(row)

summary = pd.DataFrame(rows).set_index("trial")

# Numeric columns eligible for gradient colouring.
numeric_cols = [c for c in summary.columns
                if c != "dilation_schedule" and pd.api.types.is_numeric_dtype(summary[c])]

summary.head(10).style \
    .background_gradient(subset=["val_loss"], cmap="RdYlGn_r") \
    .background_gradient(subset=[c for c in numeric_cols if c != "val_loss"], cmap="Blues")

---
# Leave-one-athlete-out CV — multi-trial comparison

In [ ]:
import json, glob

loao_files = sorted(glob.glob(os.path.join(LOAO_DIR, "loao_trial_*.json")))
if not loao_files:
    raise FileNotFoundError(f"No loao_trial_*.json files in {LOAO_DIR}")

loao_trials = []
for path in loao_files:
    with open(path) as f:
        loao_trials.append(json.load(f))
loao_trials.sort(key=lambda d: d["trial_id"])

CLASS_NAMES = ["left_stance", "right_stance", "flight"]
EVENT_KEYS  = ["left_landing", "left_takeoff", "right_landing", "right_takeoff"]

def _fold_timing_mean(trial_data, event_key):
    vals = [fold["timing_error"][event_key]["ms"]
            for fold in trial_data["folds"]
            if not np.isnan(fold["timing_error"][event_key]["ms"])]
    return float(np.mean(vals)) if vals else float("nan")

rows = []
for d in loao_trials:
    row = {
        "trial_id":      d["trial_id"],
        "macro_f1_mean": d["macro_f1_mean"],
        "macro_f1_std":  d["macro_f1_std"],
        **{f"f1_{cls}": d["per_class_f1_mean"][cls] for cls in CLASS_NAMES},
        **{ek + "_ms": _fold_timing_mean(d, ek) for ek in EVENT_KEYS},
        **d["params"],
    }
    rows.append(row)

loao_df = pd.DataFrame(rows).set_index("trial_id")

print(f"Trials loaded: {len(loao_trials)}\n")
print(f"{'Trial':>6}  {'Macro F1':>10}  {'Schedule':<14}  {'n_blocks':>8}  {'n_filters':>9}  {'kernel':>6}")
print("-" * 62)
for d in loao_trials:
    p = d["params"]
    print(f"  #{d['trial_id']:<4}  {d['macro_f1_mean']:.3f} ± {d['macro_f1_std']:.3f}  "
          f"{p.get('dilation_schedule','?'):<14}  {p['n_blocks']:>8}  {p['n_filters']:>9}  {p['kernel_size']:>6}")

## Macro F1 comparison across trials

In [ ]:
schedules     = loao_df["dilation_schedule"].values if "dilation_schedule" in loao_df.columns else [""] * len(loao_df)
schedule_set  = sorted(set(schedules))
palette       = dict(zip(schedule_set, cm.tab10(np.linspace(0, 0.8, len(schedule_set)))))
bar_colors    = [palette[s] for s in schedules]

df_sorted = loao_df.sort_values("macro_f1_mean", ascending=False)
labels    = [f"#{tid}" for tid in df_sorted.index]
x         = np.arange(len(df_sorted))

fig, ax = plt.subplots(figsize=(max(8, len(df_sorted) * 0.9), 5))
bars = ax.bar(x, df_sorted["macro_f1_mean"],
              yerr=df_sorted["macro_f1_std"],
              capsize=4, width=0.6, alpha=0.85,
              color=[palette[s] for s in df_sorted["dilation_schedule"]] if "dilation_schedule" in df_sorted.columns else "#4e79a7")

ax.set_xticks(x)
ax.set_xticklabels(
    [f"#{tid}\n{df_sorted.loc[tid, 'dilation_schedule'] if 'dilation_schedule' in df_sorted.columns else ''}"
     for tid in df_sorted.index],
    fontsize=9,
)
ax.set_ylabel("Macro F1")
ax.set_title("Macro F1 mean ± std per LOAO trial (sorted by F1)")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)

for bar, val in zip(bars, df_sorted["macro_f1_mean"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=8)

# legend for dilation schedule
if "dilation_schedule" in df_sorted.columns:
    handles = [plt.Rectangle((0, 0), 1, 1, color=palette[s], alpha=0.85) for s in schedule_set]
    ax.legend(handles, schedule_set, title="dilation schedule", fontsize=9, loc="lower right")

plt.tight_layout()
plt.show()

## Per-class F1 heatmap

In [ ]:
class_labels = ["Left stance", "Right stance", "Flight"]
f1_mat = loao_df[[f"f1_{c}" for c in CLASS_NAMES]].sort_values("f1_left_stance", ascending=False)
trial_labels = [f"#{tid}\n({loao_df.loc[tid, 'dilation_schedule'] if 'dilation_schedule' in loao_df.columns else ''})"
                for tid in f1_mat.index]

fig, ax = plt.subplots(figsize=(6, max(4, len(f1_mat) * 0.45)))
im = ax.imshow(f1_mat.values, cmap="RdYlGn", vmin=0.5, vmax=1.0, aspect="auto")

ax.set_xticks(range(len(class_labels)))
ax.set_xticklabels(class_labels)
ax.set_yticks(range(len(f1_mat)))
ax.set_yticklabels(trial_labels, fontsize=8)
ax.set_title("Per-class F1 by trial")

for i in range(len(f1_mat)):
    for j in range(len(class_labels)):
        val = f1_mat.values[i, j]
        ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=8,
                color="black")

plt.tight_layout()
plt.show()

## Confusion matrices

In [ ]:
n_trials = TOP_N
cm_class_names = ["Left\nstance", "Right\nstance", "Flight"]
ncols = min(4, n_trials)
nrows = (n_trials + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
axes = np.array(axes).flatten()

for ax, d in zip(axes, loao_trials):
    cm_raw  = np.array(d["total_confusion_matrix"], dtype=float)
    cm_norm = cm_raw / cm_raw.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(cm_class_names, fontsize=7)
    ax.set_yticklabels(cm_class_names, fontsize=7)
    ax.set_xlabel("Predicted", fontsize=7)
    ax.set_ylabel("True", fontsize=7)
    sched = d["params"].get("dilation_schedule", "")
    ax.set_title(f"Trial #{d['trial_id']} ({sched})\nF1={d['macro_f1_mean']:.3f}", fontsize=8)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if cm_norm[i, j] > 0.5 else "black")

for ax in axes[n_trials:]:
    ax.set_visible(False)

plt.suptitle("Aggregate confusion matrices (row-normalised) per trial", y=1.01)
plt.tight_layout()
plt.show()

## Summary table

In [ ]:
param_cols  = [c for c in loao_df.columns if c in {"lr", "dropout", "n_blocks", "n_filters", "kernel_size", "dilation_schedule"}]
display_df  = loao_df[["macro_f1_mean", "macro_f1_std",
                        "f1_left_stance", "f1_right_stance", "f1_flight",
                        *param_cols]].copy()

display_df.index = [f"#{tid}" for tid in display_df.index]
display_df.columns = (
    ["macro F1", "F1 std", "left F1", "right F1", "flight F1"]
    + param_cols
)

numeric_grad = ["macro F1", "left F1", "right F1", "flight F1"]

display_df.sort_values("macro F1", ascending=False).style \
    .background_gradient(subset=numeric_grad, cmap="RdYlGn") \
    .format({
        "macro F1": "{:.3f}", "F1 std": "{:.3f}",
        "left F1": "{:.3f}", "right F1": "{:.3f}", "flight F1": "{:.3f}",
        "lr": "{:.2e}",
    })